### Build races dimension
1. Read "silver" constructors table
2. Read "gold" ref_nationality_table table
3. Join data from "constructors" with "ref_nationality_region" using "nationality"
4. Select required columns
    -   constructors.constructor_id
    -   constructors.constructors_name
    -   constructors.nationality
    -   ref_nationality_region.region
7. Write transformed data to gold dim_constructors table

In [0]:
%run ../Workspace/common/configuration_environment

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_constructors"

##### Read source tables
Read "silver" constructors table, Read "gold" ref_nationality_table table

In [0]:
constructors_df = spark.table(f"{catalog_name}.{silver_schema}.constructors")
ref_nationality_region_df = spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")


##### Join data from "constructors" with "ref_nationality_region" using "nationality"
    Select required columns
    -   constructors.constructor_id
    -   constructors.constructor_name
    -   constructors.nationality
    -   ref_nationality_region.region

In [0]:
dim_constructors_df = (
    constructors_df
    .join(
            ref_nationality_region_df,
            constructors_df.nationality == ref_nationality_region_df.nationality,
            "left"
        )
    .select(
        constructors_df.constructor_id,
        constructors_df.constructor_name,
        constructors_df.nationality,
        ref_nationality_region_df.region
    )
    
)

display(dim_constructors_df)

constructor_id,constructor_name,nationality,region
adams,Adams,American,North America
afm,AFM,German,Europe
ags,AGS,French,Europe
alfa,Alfa Romeo,Swiss,Europe
alphatauri,AlphaTauri,Italian,Europe
alpine,Alpine F1 Team,French,Europe
alta,Alta,British,Europe
amon,Amon,New Zealander,Oceania
apollon,Apollon,Swiss,Europe
arrows,Arrows,British,Europe


##### Write transformed data to gold dim_constructors table

In [0]:
(
    dim_constructors_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)    
)

display(spark.table(target_table))

constructor_id,constructor_name,nationality,region
adams,Adams,American,North America
afm,AFM,German,Europe
ags,AGS,French,Europe
alfa,Alfa Romeo,Swiss,Europe
alphatauri,AlphaTauri,Italian,Europe
alpine,Alpine F1 Team,French,Europe
alta,Alta,British,Europe
amon,Amon,New Zealander,Oceania
apollon,Apollon,Swiss,Europe
arrows,Arrows,British,Europe
